# 📓 Notebook 2 of 3 — ResNet-18 Trained From Scratch

**Part of the project:** *Self-Supervised (DINOv2) vs. Supervised CNN Representations for Galaxy Morphology Classification*

| | |
|---|---|
| **Needs** | Notebook 1 finished, with its files in `MyDrive/galaxy_project/<full or quick>/` |
| **This notebook does** | Reloads the dataset and splits from Drive, then trains **Model 3 — ResNet-18 from scratch** at every label fraction × every seed |
| **Steps covered** | 1 (short), 5 (reload), 6d, 7a, 7b (model 3) |
| **Saves to Google Drive** | `records_nb2_scratch.pkl` — results of models 1, 2 **and** 3 (cumulative) |
| **Next** | ➡️ **Notebook 3** (ResNet-18 ImageNet, fine-tuned DINOv2, and all the analysis) |
| **Runtime** | Google Colab, **T4 GPU**. This is a long notebook; it saves after **every** run, so a disconnect only costs the run in progress |

**Why does this notebook exist?** Training a whole CNN from random weights, with augmentation, at 6 label fractions × 5 seeds is the single slowest part of the study (30 full training runs). Giving it a notebook of its own keeps every session short enough for free Colab.

**If Colab disconnects:** reconnect, run the notebook again from the top. Finished runs are loaded from Drive and skipped.

## Step 1 — Setup (short version)

**Why?** A new session starts empty: no libraries loaded, no data, no settings. **What?** Load the libraries, connect Google Drive, read `config.json` (the settings fixed in Notebook 1) and choose the GPU. **How?** The only value to set by hand is `RUN_NAME`, which must equal the mode used in Notebook 1. `MAX_SESSION_MINUTES` is optional: if you know your session will not last long, set it (for example to `240`) and the loop stops cleanly instead of being cut off mid-run.

In [1]:
import os, time, random, math, json, gc, pickle, shutil, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if DEVICE.type != "cuda":
    print("⚠️  No GPU found. Use Runtime -> Change runtime type -> T4 GPU, then run this notebook again.")

# ============================ SETTINGS FOR THIS NOTEBOOK ============================
RUN_NAME = "full"            # MUST match Notebook 1:  "quick" if QUICK_MODE was True there, else "full"
MAX_SESSION_MINUTES = None   # optional graceful stop, e.g. 240 = stop starting new runs after 4 hours
# ====================================================================================

# ---- Google Drive: this is where Notebook 1's results are waiting ----
from google.colab import drive
drive.mount("/content/drive")
PROJECT_DIR = f"/content/drive/MyDrive/galaxy_project/{RUN_NAME}"
FIG_DIR     = f"{PROJECT_DIR}/figures"
RES_DIR     = f"{PROJECT_DIR}/results"
for d in (FIG_DIR, RES_DIR):
    os.makedirs(d, exist_ok=True)

CFG_PATH = f"{PROJECT_DIR}/config.json"
assert os.path.exists(CFG_PATH), (
    f"{CFG_PATH} not found. Run Notebook 1 first, and check that RUN_NAME ('{RUN_NAME}') "
    f"matches the mode used there ('quick' or 'full').")
with open(CFG_PATH) as f:
    cfg = json.load(f)
assert cfg.get("notebook1_finished"), "Notebook 1 did not finish -- run it to the end first."

# Every setting below is READ from Notebook 1 -- do not change them here.
QUICK_MODE   = cfg["quick_mode"]
SEEDS        = cfg["seeds"]
FRACTIONS    = cfg["fractions"]
CNN_EPOCHS   = cfg["cnn_epochs"]
FT_EPOCHS    = cfg["ft_epochs"]
PROBE_EPOCHS = cfg["probe_epochs"]
CLASS_NAMES  = cfg["class_names"]
N_CLASSES    = len(CLASS_NAMES)
IMG_SIZE     = cfg["img_size"]
CLASS_WEIGHTS = torch.tensor(cfg["class_weights"], dtype=torch.float32).to(DEVICE)
print(f"Loaded config: QUICK_MODE={QUICK_MODE} | seeds={SEEDS} | fractions={FRACTIONS} | "
      f"CNN epochs={CNN_EPOCHS} | FT epochs={FT_EPOCHS}")

Using device: cuda
Mounted at /content/drive
Loaded config: QUICK_MODE=False | seeds=[0, 1, 2, 3, 4] | fractions=[0.01, 0.05, 0.1, 0.25, 0.5, 1.0] | CNN epochs=15 | FT epochs=8


In [2]:
# ---- Helpers that make every experiment loop RESUMABLE ---------------------------------------
# After every finished run the full list of results ("records") is written to Google Drive.
# If Colab disconnects, re-running the notebook reloads that file and skips all finished runs.

def save_records(records, path):
    tmp = path + ".tmp"                      # write to a temporary file first, so that a disconnect
    with open(tmp, "wb") as f:               # in the middle of writing cannot corrupt the real file
        pickle.dump(records, f)
    try:
        os.replace(tmp, path)
    except OSError:
        shutil.copyfile(tmp, path); os.remove(tmp)

def load_records(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def run_key(model, fraction, seed):
    # A run is identified by (model, label fraction, seed).
    return (model, round(float(fraction), 6), int(seed))

def done_keys(records):
    return {run_key(r["model"], r["fraction"], r["seed"]) for r in records}

def quick_preview(records):
    # Mean / std / count of test accuracy per (model, fraction) -- a quick look at progress.
    rows = [dict(model=r["model"], fraction=r["fraction"], seed=r["seed"],
                 accuracy=accuracy_score(r["y_true"], r["preds"])) for r in records]
    return (pd.DataFrame(rows).groupby(["model", "fraction"])["accuracy"]
              .agg(["mean", "std", "count"]).round(4))


# Optional graceful stop: when MAX_SESSION_MINUTES is set, the loops below stop *before starting*
# a new run once that many minutes have passed. Everything finished so far is already on Drive,
# so you can simply run the notebook again in a fresh session to continue.
SESSION_START = time.time()

class TimeBudgetReached(Exception):
    pass

def out_of_time():
    return MAX_SESSION_MINUTES is not None and (time.time() - SESSION_START) / 60 > MAX_SESSION_MINUTES


# Small helpers (defined in Step 6a of Notebook 1, repeated here because this is a fresh session).
# Freezing = switching off gradient updates, so the weights stay exactly as pretrained.
def freeze(model):
    for p in model.parameters():
        p.requires_grad = False

def count_params(model, trainable_only=True):
    return sum(p.numel() for p in model.parameters() if (p.requires_grad or not trainable_only))

## Step 5 (reload) — Dataset, transforms and splits from Notebook 1

**Why?** Every model must be trained and tested on exactly the same images. **What?** The image zip is copied from Drive to Colab's local disk and unpacked (reading thousands of small files directly from Drive would be very slow), the train / val / test tables are rebuilt from `working_dataset.csv`, and the **same** transforms and `Dataset` class as in Notebook 1 are defined. **How?** A safety check compares the rebuilt labels with those stored beside the embeddings; if anything differs, the notebook stops rather than silently producing inconsistent results.

The transforms are unchanged: resize to 224 × 224, ImageNet normalisation, and — for training only — random horizontal flip and ±15° rotation (no colour jitter).

In [3]:
# ---- Rebuild the SAME dataset and splits that Notebook 1 created ------------------------------
IMG_ROOT   = "/content/gz2_subset"                          # local (fast) copy of the images
ZIP_DRIVE  = f"{PROJECT_DIR}/gz2_subset_images.zip"
ds_all     = pd.read_csv(f"{PROJECT_DIR}/working_dataset.csv")   # columns: img_rel, label, split

def images_ready():
    rels = list(ds_all["img_rel"].iloc[::200]) + [ds_all["img_rel"].iloc[0], ds_all["img_rel"].iloc[-1]]
    return all(os.path.exists(os.path.join(IMG_ROOT, p)) for p in rels)

if not images_ready():
    t0 = time.time()
    shutil.copyfile(ZIP_DRIVE, "/content/gz2_subset_images.zip")      # Drive -> local disk
    with zipfile.ZipFile("/content/gz2_subset_images.zip") as zf:
        zf.extractall(IMG_ROOT)
    os.remove("/content/gz2_subset_images.zip")
    print(f"Unpacked {len(ds_all)} images to {IMG_ROOT} in {time.time()-t0:.0f}s")
assert images_ready(), "Image files are missing -- check gz2_subset_images.zip on Drive."

ds_all["img_path"] = IMG_ROOT + "/" + ds_all["img_rel"]
train_df = ds_all[ds_all["split"] == "train"].reset_index(drop=True)
val_df   = ds_all[ds_all["split"] == "val"].reset_index(drop=True)
test_df  = ds_all[ds_all["split"] == "test"].reset_index(drop=True)
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

# Safety check: the labels must line up exactly with the embeddings saved by Notebook 1.
emb = np.load(f"{PROJECT_DIR}/embeddings.npz")
assert np.array_equal(train_df["label"].values, emb["train_labels_full"]), "train split differs from Notebook 1!"
assert np.array_equal(val_df["label"].values,   emb["val_labels"]),        "val split differs from Notebook 1!"
assert np.array_equal(test_df["label"].values,  emb["test_labels"]),       "test split differs from Notebook 1!"
print("✅ Splits match Notebook 1 exactly.")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Reads one image from disk, applies the chosen transform and returns (image tensor, class label).
class GalaxyImageDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True); self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self.transform(Image.open(row["img_path"]).convert("RGB"))
        return img, int(row["label"])

train_ds_aug   = GalaxyImageDataset(train_df, train_transform)
train_ds_plain = GalaxyImageDataset(train_df, eval_transform)
val_ds         = GalaxyImageDataset(val_df, eval_transform)
test_ds        = GalaxyImageDataset(test_df, eval_transform)
train_labels_full = train_df["label"].values

Unpacked 20000 images to /content/gz2_subset in 20s
train=14000  val=3000  test=3000
✅ Splits match Notebook 1 exactly.


### Step 6d — ResNet-18 baselines
**Why?** They represent the supervised alternative (models 3 and 4). **What?** A ResNet-18 whose final layer is replaced to give 5 outputs; `pretrained=False` starts from random weights, `pretrained=True` starts from ImageNet weights. **How?** `train_cnn()` trains the whole network end to end (Adam, learning rate 1e-4, class-weighted loss) and records training time, inference time per image and peak GPU memory.

In [4]:
# ResNet-18 whose last layer is replaced to output 5 classes.
# pretrained=True starts from ImageNet weights; pretrained=False starts from random weights.
def build_resnet18(n_classes, pretrained):
    weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = torchvision.models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, n_classes)
    return model.to(DEVICE)

# Train the whole CNN end to end and record training time, inference time and peak GPU memory.
def train_cnn(model, train_dataset, eval_dataset, epochs, batch_size=64, lr=1e-4, seed=0, class_weights=None):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss(weight=class_weights)
    g = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, generator=g)
    eval_loader  = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for _ in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
    train_time = time.time() - t0
    peak_mem_mb = torch.cuda.max_memory_allocated() / 1e6 if DEVICE.type == "cuda" else 0.0

    model.eval()
    all_preds, all_labels = [], []
    t0 = time.time()
    with torch.no_grad():
        for xb, yb in eval_loader:
            preds = model(xb.to(DEVICE)).argmax(1).cpu().numpy()
            all_preds.append(preds); all_labels.append(yb.numpy())
    infer_time_per_img = (time.time() - t0) / max(1, len(eval_dataset))
    return model, np.concatenate(all_preds), np.concatenate(all_labels), train_time, infer_time_per_img, peak_mem_mb

### Step 7a — Stratified subsampling helper
**Why?** With plain random sampling, a small class such as Edge-on disk could receive *zero* examples at 1% labels, silently turning the experiment into a 4-class problem. **What?** A function that returns the indices of a labelled subset. **How?** `stratified_subsample()` samples the chosen fraction *within each class* (at least one image per class), using a seed derived from the run's seed and fraction.

In [5]:
def stratified_subsample(labels_array, frac, seed, n_classes):
    # Stratified (per-class) subsampling for the label-efficiency sweep. Plain random sampling of a
    # tiny class such as Edge-on disk could easily pick ZERO examples of it at frac=0.01, quietly
    # turning that experiment into a 4-class problem. Sampling a fraction from WITHIN each class
    # guarantees that every class appears (at least 1 example) at every label fraction, which keeps
    # the label-efficiency curve comparable across fractions and consistent with the class-weighted loss.
    rng = np.random.RandomState(seed)
    sub_idx = []
    for cls in range(n_classes):
        cls_idx = np.where(labels_array == cls)[0]
        if len(cls_idx) == 0:
            continue
        n_c = len(cls_idx) if frac >= 1.0 else max(1, int(round(len(cls_idx) * frac)))
        n_c = min(n_c, len(cls_idx))
        sub_idx.append(rng.choice(cls_idx, size=n_c, replace=False))
    sub_idx = np.concatenate(sub_idx)
    rng.shuffle(sub_idx)
    return sub_idx

### Step 7b (model 3) — ResNet-18 from scratch at every fraction
**Why?** It is the supervised baseline without any pretraining, and the main comparator for the frozen DINOv2 probe (RQ1, RQ2).
**What?** For every label fraction × every seed (in the full run: 6 × 5 = 30 runs), a ResNet-18 with random initial weights is trained end to end and evaluated on the fixed test set. It receives **the same labelled subset** as the two probes in Notebook 1 for that (fraction, seed).
**How?** The results of Notebook 1 are loaded first, and every new result is appended to them, so `records_nb2_scratch.pkl` always holds models 1, 2 and 3 together. The list is saved to Drive after every run. Runs already present are skipped, so the cell can be re-run after a disconnect.

> **Time note.** Training time grows with the number of labelled images, so the 100% and 50% runs take by far the longest. Each run prints its own duration; use the first 100% run to estimate how long the whole notebook will take.

In [6]:
REC_NB1 = f"{PROJECT_DIR}/records_nb1_probes.pkl"
REC_NB2 = f"{PROJECT_DIR}/records_nb2_scratch.pkl"

if os.path.exists(REC_NB2):
    records = load_records(REC_NB2)
    print(f"Resuming: loaded {len(records)} records from {os.path.basename(REC_NB2)}")
else:
    records = load_records(REC_NB1)
    print(f"Starting: loaded {len(records)} probe records from Notebook 1")

n_probe = sum(r["model"].startswith("DINOv2-frozen") for r in records)
assert n_probe == cfg["probe_records_expected"], "Notebook 1's probe results are incomplete -- finish Notebook 1 first."

MODEL_SCRATCH = "ResNet18-scratch"
done = done_keys(records)
total_runs = len(FRACTIONS) * len(SEEDS)
n_done = sum(1 for k in done if k[0] == MODEL_SCRATCH)
print(f"{MODEL_SCRATCH}: {n_done} / {total_runs} runs already finished")

try:
    for frac in FRACTIONS:
        for seed in SEEDS:
            key = run_key(MODEL_SCRATCH, frac, seed)
            if key in done:
                continue
            if out_of_time():
                raise TimeBudgetReached
            t_run = time.time()
            sub_idx = stratified_subsample(train_labels_full, frac, seed=seed * 1000 + int(frac * 100), n_classes=N_CLASSES)
            n_sub = len(sub_idx)

            cnn = build_resnet18(N_CLASSES, pretrained=False)
            cnn_sub_ds = Subset(train_ds_aug, sub_idx.tolist())
            _, preds, y_t, ttime, itime, mem = train_cnn(cnn, cnn_sub_ds, test_ds, CNN_EPOCHS, seed=seed,
                                                           class_weights=CLASS_WEIGHTS)
            assert np.array_equal(y_t, test_df["label"].values)      # same test set, same order, as Notebook 1
            records.append(dict(model=MODEL_SCRATCH, fraction=frac, seed=seed, n_labeled=n_sub,
                                 preds=preds, y_true=y_t, train_time=ttime, infer_time=itime,
                                 peak_mem_mb=mem, n_params=count_params(cnn)))
            done.add(key); n_done += 1
            save_records(records, REC_NB2)                       # <- progress is now safe on Drive

            acc = accuracy_score(y_t, preds)
            print(f"[frac={frac:.2f} seed={seed}] n_labeled={n_sub} acc={acc:.4f} "
                  f"({time.time()-t_run:.0f}s)  -> {n_done}/{total_runs} scratch runs done")
            del cnn, cnn_sub_ds; gc.collect()
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
except TimeBudgetReached:
    print(f"\n⏸️  Time budget of {MAX_SESSION_MINUTES} min reached. Progress is saved on Drive -- "
          f"run this notebook again in a new session to continue.")

Starting: loaded 60 probe records from Notebook 1
ResNet18-scratch: 0 / 30 runs already finished
[frac=0.01 seed=0] n_labeled=140 acc=0.3190 (37s)  -> 1/30 scratch runs done
[frac=0.01 seed=1] n_labeled=140 acc=0.3693 (28s)  -> 2/30 scratch runs done
[frac=0.01 seed=2] n_labeled=140 acc=0.3610 (27s)  -> 3/30 scratch runs done
[frac=0.01 seed=3] n_labeled=140 acc=0.3060 (27s)  -> 4/30 scratch runs done
[frac=0.01 seed=4] n_labeled=140 acc=0.2940 (27s)  -> 5/30 scratch runs done
[frac=0.05 seed=0] n_labeled=700 acc=0.4593 (63s)  -> 6/30 scratch runs done
[frac=0.05 seed=1] n_labeled=700 acc=0.5533 (62s)  -> 7/30 scratch runs done
[frac=0.05 seed=2] n_labeled=700 acc=0.4470 (62s)  -> 8/30 scratch runs done
[frac=0.05 seed=3] n_labeled=700 acc=0.5543 (63s)  -> 9/30 scratch runs done
[frac=0.05 seed=4] n_labeled=700 acc=0.4630 (63s)  -> 10/30 scratch runs done
[frac=0.10 seed=0] n_labeled=1400 acc=0.4683 (109s)  -> 11/30 scratch runs done
[frac=0.10 seed=1] n_labeled=1400 acc=0.3033 (111s) 

### Progress check and preview
**Why?** To know whether Notebook 2 is really finished before moving on. **What?** The number of finished ResNet-18 (scratch) runs against the number expected, and the mean accuracy of models 1 – 3 so far. If the notebook is complete the cell says so; if not, re-run the loop cell above (it continues from where it stopped).

In [7]:
n_scratch = sum(r["model"] == MODEL_SCRATCH for r in records)
display(quick_preview(records))
if n_scratch == total_runs:
    print(f"\n✅ Notebook 2 complete: {n_scratch}/{total_runs} ResNet-18 (scratch) runs saved in {REC_NB2}")
    print("➡️  Now open Notebook 3 and set RUN_NAME =", repr(RUN_NAME), "there.")
else:
    print(f"\n⚠️  Only {n_scratch}/{total_runs} ResNet-18 (scratch) runs are finished. "
          f"Re-run the loop cell above (in this or a new session) before opening Notebook 3.")

mean     std  count
model                fraction                       
DINOv2-frozen+Linear 0.01      0.5348  0.0104      5
                     0.05      0.6168  0.0139      5
                     0.10      0.6513  0.0062      5
                     0.25      0.6741  0.0051      5
                     0.50      0.6895  0.0063      5
                     1.00      0.6997  0.0068      5
DINOv2-frozen+MLP    0.01      0.5623  0.0072      5
                     0.05      0.6191  0.0191      5
                     0.10      0.6439  0.0084      5
                     0.25      0.6720  0.0101      5
                     0.50      0.6832  0.0102      5
                     1.00      0.6959  0.0113      5
ResNet18-scratch     0.01      0.3299  0.0335      5
                     0.05      0.4954  0.0537      5
                     0.10      0.4723  0.1005      5
                     0.25      0.5440  0.0473      5
                     0.50      0.6961  0.1231      5
                     1.00      0.6866  0.1856      5


✅ Notebook 2 complete: 30/30 ResNet-18 (scratch) runs saved in /content/drive/MyDrive/galaxy_project/full/records_nb2_scratch.pkl
➡️  Now open Notebook 3 and set RUN_NAME = 'full' there.


### Final cell — flush Google Drive
Drive writes are buffered. Running this cell (last) forces everything to be written before you close the tab. After it runs, Drive is unmounted — that is expected.

In [8]:
from google.colab import drive
drive.flush_and_unmount()
print("Drive flushed. You can now close this notebook and open Notebook 3.")

Drive flushed. You can now close this notebook and open Notebook 3.
